# **Configurações — Parte 2 (meses 07 a 12)**

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS fiap

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS fiap.bronze;
CREATE SCHEMA IF NOT EXISTS fiap.silver;
CREATE VOLUME IF NOT EXISTS fiap.bronze.arquivos_vacinacao;

# Ingestão **PNI - Vacinação (SP)** — Parte 2

**Processa só os meses 07 a 12 de 2025.**
A etapa de streaming filtra a vacina de dengue (Qdenga) e a UF 

In [0]:
%pip install ijson
dbutils.library.restartPython()

In [0]:
import os
import gc
import json
import ijson
import zipfile
import requests
from pyspark.sql.functions import col, trim, upper, substring, to_date, coalesce, year, month, current_timestamp, lit

ANOS = [2025]
TODOS_OS_MESES = {
    1: "jan", 2: "fev", 3: "mar", 4: "abr", 5: "mai", 6: "jun",
    7: "jul", 8: "ago", 9: "set", 10: "out", 11: "nov", 12: "dez",
}
MESES = {k: v for k, v in TODOS_OS_MESES.items() if 7 <= k <= 12}

UF = "SP"
CATALOGO = "fiap"
TABELA_DESTINO = f"{CATALOGO}.bronze.PNI_VACINACAO_{UF}"
volume_path = f"/Volumes/{CATALOGO}/bronze/arquivos_vacinacao"

# Campo de UF do paciente confirmado num registro real ("sigla_uf_paciente").
# Mantido fallback pro padrao abreviado como seguranca extra, sem custo.
CAMPOS_UF_PACIENTE = ["sigla_uf_paciente", "sg_uf_paciente"]
CAMPOS_VACINA = ["sigla_vacina", "sg_vacina"]


def obter_campo(registro, candidatos):
    for c in candidatos:
        valor = registro.get(c)
        if valor not in (None, ""):
            return valor
    return None


print(f"Verificando historico {TABELA_DESTINO}")
try:
    df_existente = spark.table(TABELA_DESTINO)
    periodos_processados = set(
        row["PERIODO_REFERENCIA"] for row in df_existente.select("PERIODO_REFERENCIA").distinct().collect()
    )
except Exception:
    print("Tabela nao encontrada ou vazia. Iniciando carga")
    periodos_processados = set()

os.makedirs("/tmp/vacinacao_download/", exist_ok=True)
dbutils.fs.mkdirs(f"dbfs:{volume_path}")

primeira_carga = len(periodos_processados) == 0
primeiro_mes_processado = True

for ano in ANOS:
    for mes_num, mes_abrev in MESES.items():
        periodo_ref = f"{ano}-{mes_num:02d}"
        print(f"\nProcessando {periodo_ref}")

        if periodo_ref in periodos_processados:
            print(f"pulando {periodo_ref} - ja existe na tabela")
            continue

        diretorio_local = f"/tmp/vacinacao_{ano}_{mes_num:02d}/"
        os.makedirs(diretorio_local, exist_ok=True)
        arquivo_zip = os.path.join(diretorio_local, f"vacinacao_{periodo_ref}.zip")

        try:
            url = f"https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/json/vacinacao_{mes_abrev}_{ano}_json.zip"

            print(f"Baixando {periodo_ref} (streaming para disco, pode ser alguns GB)")
            with requests.get(url, stream=True, timeout=300) as r:
                if r.status_code != 200:
                    print(f"Aviso: HTTP {r.status_code} para {periodo_ref}, pulando")
                    continue
                with open(arquivo_zip, "wb") as f_out:
                    for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                        f_out.write(chunk)

            if os.path.getsize(arquivo_zip) == 0:
                print(f"Aviso: arquivo vazio para {periodo_ref}, pulando")
                continue

            print(f"Descompactando {periodo_ref}")
            with zipfile.ZipFile(arquivo_zip) as zf:
                nomes_json = [n for n in zf.namelist() if n.endswith(".json")]
                if not nomes_json:
                    print(f"Aviso: nenhum .json dentro do zip de {periodo_ref}, pulando")
                    continue
                nome_json = nomes_json[0]
                zf.extract(nome_json, diretorio_local)

            caminho_json = os.path.join(diretorio_local, nome_json)

            # streaming: le token a token e ja filtra SP + dengue no mesmo passo -
            # nunca materializa o Brasil inteiro (nem SP inteiro com outras vacinas) em outro arquivo
            caminho_filtrado = os.path.join(diretorio_local, f"sp_dengue_{periodo_ref}.jsonl")
            total_lidos = 0
            total_sp = 0
            total_sp_dengue = 0
            with open(caminho_json, "rb") as f_in, open(caminho_filtrado, "w", encoding="utf-8") as f_out:
                for registro in ijson.items(f_in, "item"):
                    total_lidos += 1
                    uf_valor = obter_campo(registro, CAMPOS_UF_PACIENTE)
                    if not uf_valor or str(uf_valor).strip().upper() != UF:
                        continue
                    total_sp += 1
                    vacina_valor = str(obter_campo(registro, CAMPOS_VACINA) or "").strip().upper()
                    if "DENGUE" in vacina_valor or "QDENGA" in vacina_valor:
                        f_out.write(json.dumps(registro, ensure_ascii=False) + "\n")
                        total_sp_dengue += 1
                    if total_lidos % 200000 == 0:
                        print(f"  ... {total_lidos:,.0f} lidos (Brasil), {total_sp:,.0f} de SP, "
                              f"{total_sp_dengue:,.0f} de SP+dengue ate agora")

            print(f"{periodo_ref}: {total_lidos:,.0f} lidos (Brasil), {total_sp:,.0f} de SP, "
                  f"{total_sp_dengue:,.0f} de SP+dengue")
            total_sp = total_sp_dengue

            # libera espaco: apaga zip e json bruto do Brasil, mantem so o filtrado 
            os.remove(arquivo_zip)
            os.remove(caminho_json)

            if total_sp == 0:
                print(f"Aviso: 0 registros de SP em {periodo_ref}")
                os.remove(caminho_filtrado)
                continue

            caminho_destino = f"{volume_path}/sp_{periodo_ref}.jsonl"
            resultado_cp = os.system(f"cp {caminho_filtrado} {caminho_destino}")

            # confere que a copia pro Volume realmente funcionou antes de apagar o local e ler
            if resultado_cp != 0 or not os.path.exists(caminho_destino) or os.path.getsize(caminho_destino) == 0:
                print(f"AVISO: copia para o Volume falhou ou ficou vazia (codigo {resultado_cp}). "
                      f"Lendo direto do disco local do driver como alternativa (prefixo file:).")
                caminho_leitura = f"file:{caminho_filtrado}"
            else:
                caminho_leitura = caminho_destino
                os.remove(caminho_filtrado)

            df_mes = spark.read.format("json").load(caminho_leitura)

            if primeiro_mes_processado:
                print(f"\nSchema do primeiro mes processado ({periodo_ref}) - conferir colunas:")
                df_mes.printSchema()
                primeiro_mes_processado = False

            df_mes = df_mes \
                .withColumn("ANO_REFERENCIA", lit(ano)) \
                .withColumn("MES_REFERENCIA", lit(mes_num)) \
                .withColumn("PERIODO_REFERENCIA", lit(periodo_ref)) \
                .withColumn("DATA_CARGA", current_timestamp())

            modo = "overwrite" if primeira_carga else "append"
            df_mes.write.format("delta").mode(modo).option("mergeSchema", "true").saveAsTable(TABELA_DESTINO)
            primeira_carga = False
            periodos_processados.add(periodo_ref)
            print(f"Periodo {periodo_ref} salvo com sucesso ({total_sp:,.0f} registros de SP)")

            del df_mes
            gc.collect()

        except Exception as e:
            print(f"ERRO no periodo {periodo_ref}: {e}")

print("\nOK Ingestao do PNI (Bronze) finalizada para esta parte!")

## Conferência rápida

In [0]:
%sql
SELECT ano_referencia, mes_referencia, COUNT(*) AS doses_aplicadas
FROM fiap.bronze.PNI_VACINACAO_SP
GROUP BY ano_referencia, mes_referencia
ORDER BY ano_referencia, mes_referencia